In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

In [ ]:
PROJECT_ROOT = Path("/path/to/ainu_root")
IMAGE_PATH = PROJECT_ROOT / "paper/human_data/"
output_dir = PROJECT_ROOT / "final_dataset" / "features_analysis" / "new_grouping3"
images_features = Path(PROJECT_ROOT / "cell_features.csv")
probability_file = Path(PROJECT_ROOT / "final_dataset" / "results" / "embeddings" / "embeddings_samlpe4_split3.csv")

features_df = pd.read_csv(images_features, header=[0])
probability_df = pd.read_csv(probability_file, header=[0])

In [ ]:
# Merge: probability_df File == features_df File.1
df_merged = probability_df.merge(
    features_df,
    left_on="File",
    right_on="File.1",
    how="inner",
    suffixes=("_A", "_B")
)

print(df_merged.shape)
df_merged.head()

### Group by feture 

In [ ]:
import pandas as pd
import numpy as np

FEATURE_GROUPS = {
    "Size": [
        "area",
        "convex_hull_area",
        "radius_equivalent",
        "radius_circumscribed",
    ],

    "Circularity": [
        "circularity",
        "radius_inscribed_ratio",
    ],

    "Solidity": [
        "solidity",
    ],

    "Elongation / Anisotropy": [
        "eccentricity",
        "aspect_ratio",
    ],

    "Boundary irregularity": [
        "boundary_roughness",
        "boundary_smoothness",
        "radius_boundary_std",
    ],

    "Texture heterogeneity": [
        "texture_contrast",
        "texture_dissimilarity",
        "texture_homogeneity",
    ],

    "Radial intensity distribution": [
        "radial_inner_outer_ratio",
        "radial_monotonicity",
        "radial_cv",
        "radial_intensity_std",
        "radial_peak_ring",
    ],

    "Intensity asymmetry": [
        "intensity_com_offset",
    ],
}

# Keep only columns present in df
FEATURE_GROUPS = {g: [c for c in cols if c in df_merged.columns]
                  for g, cols in FEATURE_GROUPS.items()}
# Remove empty groups
FEATURE_GROUPS = {
    group: cols for group, cols in FEATURE_GROUPS.items()
    if len(cols) > 0
}
print(FEATURE_GROUPS)

In [ ]:
label_map = {
    "size": "Size",
    "circularity": "Circularity",
    "solidity": "Solidity",
    "elongation_anisotropy": "Elongation / Anisotropy",
    "boundary_irregularity": "Boundary Irregularity",
    "texture_heterogeneity": "Texture Heterogeneity",
    "radial_intensity_distribution": "Radial Intensity Distribution",
    "intensity_asymmetry": "Intensity asymmetry",
}

order = [
    "Size",
    "Circularity",
    "Solidity",
    "Elongation / Anisotropy",
    "Boundary Irregularity",
    "Texture Heterogeneity",
    "Radial Intensity Distribution",
    "Intensity asymmetry",
]

# Direct AML-vs-control comparison for each shape descriptor

### calculate the direct AML-vs-control standardized difference for each feature group by:

z-scoring each individual descriptor across all cells,
averaging descriptors within each feature group per cell,
comparing AML vs control using a Cohen’s-d-like standardized difference.

In [ ]:
patient_col = "patient"
label_col = "new_labels_A"
aml_label = "AML"
control_label = "Control"

print(df_merged["patient"].unique())
print(df_merged[label_col].value_counts())

In [ ]:
#create one score per feature group 

# All descriptor columns
all_feat_cols = [c for cols in FEATURE_GROUPS.values() for c in cols]

# Convert to numeric
X = df_merged[all_feat_cols].apply(pd.to_numeric, errors="coerce")

# Z-score each descriptor across all cells
df_z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)

# Create one mean z-score per feature group per cell
GROUP_SCORE_COLS = []

for group_name, feat_cols in FEATURE_GROUPS.items():
    clean_name = (
        group_name.lower()
        .replace(" / ", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    col_name = f"group_z_{clean_name}"

    df_merged[col_name] = df_z[feat_cols].mean(axis=1)
    GROUP_SCORE_COLS.append(col_name)

print(GROUP_SCORE_COLS)

In [ ]:
#calculate the standard difference for each group (aml vs control) 

results = []
for col in GROUP_SCORE_COLS:
    aml_values = df_merged.loc[df_merged[label_col] == aml_label, col].dropna()
    control_values = df_merged.loc[df_merged[label_col] == control_label, col].dropna()

    mean_aml = aml_values.mean()
    mean_control = control_values.mean()

    sd_aml = aml_values.std(ddof=1)
    sd_control = control_values.std(ddof=1)

    pooled_sd = np.sqrt((sd_aml**2 + sd_control**2) / 2)

    standardized_difference = (mean_aml - mean_control) / pooled_sd

    results.append({
        "feature_group": col.replace("group_z_", ""),
        "mean_AML": mean_aml,
        "mean_control": mean_control,
        "sd_AML": sd_aml,
        "sd_control": sd_control,
        "standardized_difference": standardized_difference,
        "n_AML": len(aml_values),
        "n_control": len(control_values),
    })

group_direct_comparison_df = pd.DataFrame(results)

group_direct_comparison_df = group_direct_comparison_df.sort_values(
    "standardized_difference"
)

group_direct_comparison_df.to_csv(output_dir /
    "AML_vs_CTRL_comparison.csv",
    index=False
)
group_direct_comparison_df

## Patient level comparison

In [ ]:
patient_group_df = (
    df_merged
    .groupby([patient_col, label_col])[GROUP_SCORE_COLS]
    .mean()
    .reset_index()
)

patient_group_df.head()

In [ ]:
#make comparison
patient_results = []

for col in GROUP_SCORE_COLS:
    aml_values = patient_group_df.loc[
        patient_group_df[label_col] == aml_label, col
    ].dropna()

    control_values = patient_group_df.loc[
        patient_group_df[label_col] == control_label, col
    ].dropna()

    mean_aml = aml_values.mean()
    mean_control = control_values.mean()

    sd_aml = aml_values.std(ddof=1)
    sd_control = control_values.std(ddof=1)

    pooled_sd = np.sqrt((sd_aml**2 + sd_control**2) / 2)

    standardized_difference = (mean_aml - mean_control) / pooled_sd

    patient_results.append({
        "feature_group": col.replace("group_z_", ""),
        "mean_AML": mean_aml,
        "mean_control": mean_control,
        "sd_AML": sd_aml,
        "sd_control": sd_control,
        "standardized_difference": standardized_difference,
        "n_AML_patients": len(aml_values),
        "n_control_patients": len(control_values),
    })

patient_group_direct_comparison_df = pd.DataFrame(patient_results)

patient_group_direct_comparison_df = patient_group_direct_comparison_df.sort_values(
    "standardized_difference"
)

patient_group_direct_comparison_df.to_csv(output_dir /
    "AML_vs_CTRL_patient_level_comparison.csv",
    index=False
)
patient_group_direct_comparison_df

### 
To directly compare AML and control samples for each nuclear-shape feature group, individual shape descriptors were first standardized across all cells and then averaged within predefined biological feature groups, including size, roundness/circularity, elongation, boundary roughness, and mass distribution. For each patient, the mean group-level z-score was calculated, and AML-versus-control differences were quantified using a standardized mean difference analogous to Cohen’s d, defined as the difference between AML and control means divided by the pooled standard deviation across patients.


In [ ]:
#doing statistics 
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

patient_col = "patient"
label_col = "new_labels_A"

aml_label = "AML"
control_label = "Control"

patient_results = []

for col in GROUP_SCORE_COLS:
    aml_values = patient_group_df.loc[
        patient_group_df[label_col] == aml_label, col
    ].dropna().astype(float)

    control_values = patient_group_df.loc[
        patient_group_df[label_col] == control_label, col
    ].dropna().astype(float)

    mean_aml = aml_values.mean()
    mean_control = control_values.mean()

    sd_aml = aml_values.std(ddof=1)
    sd_control = control_values.std(ddof=1)

    pooled_sd = np.sqrt((sd_aml**2 + sd_control**2) / 2)

    standardized_difference = (mean_aml - mean_control) / pooled_sd

    # Welch's t-test
    t_stat, p_value = stats.ttest_ind(
        aml_values,
        control_values,
        equal_var=False,
        nan_policy="omit"
    )

    # Mann–Whitney U test, non-parametric alternative
    u_stat, mannwhitney_p = stats.mannwhitneyu(
        aml_values,
        control_values,
        alternative="two-sided"
    )

    patient_results.append({
        "feature_group": col.replace("group_z_", ""),
        "mean_AML": mean_aml,
        "mean_control": mean_control,
        "sd_AML": sd_aml,
        "sd_control": sd_control,
        "standardized_difference": standardized_difference,
        "welch_t": t_stat,
        "welch_p": p_value,
        "mannwhitney_p": mannwhitney_p,
        "n_AML_patients": len(aml_values),
        "n_control_patients": len(control_values),
    })

patient_group_stats_df = pd.DataFrame(patient_results)

# Multiple-testing correction across feature groups
patient_group_stats_df["welch_p_adj_BH"] = multipletests(
    patient_group_stats_df["welch_p"],
    method="fdr_bh"
)[1]

patient_group_stats_df["mannwhitney_p_adj_BH"] = multipletests(
    patient_group_stats_df["mannwhitney_p"],
    method="fdr_bh"
)[1]

patient_group_stats_df = patient_group_stats_df.sort_values(
    "welch_p"
)

patient_group_stats_df.to_csv(output_dir /
    "AML_vs_CTRL_patient_level_comparison_startistics.csv",
    index=False
)

patient_group_stats_df.round(4)

In [ ]:
#plot comparisons
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

plot_df = patient_group_direct_comparison_df.copy()

plot_df["feature_group_label"] = (
    plot_df["feature_group"]
    .map(label_map)
    .fillna(plot_df["feature_group"])
)

# Add adjusted p-values from statistics dataframe
plot_df = plot_df.merge(
    patient_group_stats_df[
        ["feature_group", "welch_p_adj_BH", "mannwhitney_p_adj_BH"]
    ],
    on="feature_group",
    how="left"
)

# Choose which p-value to use for stars
# I suggest Mann–Whitney for patient-level comparison if normality is not guaranteed
p_col = "mannwhitney_p_adj_BH"

def p_to_stars(p):
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

plot_df["significance"] = plot_df[p_col].apply(p_to_stars)

# Order by standardized difference
plot_df = plot_df.sort_values("standardized_difference")

# Color by direction
colors = np.where(
    plot_df["standardized_difference"] > 0,
    "#B2182B",   # higher in AML
    "#2166AC"    # higher in control
)

fig, ax = plt.subplots(figsize=(5.5, 3))

bars = ax.barh(
    plot_df["feature_group_label"],
    plot_df["standardized_difference"],
    color=colors,
    edgecolor="black",
    linewidth=0.6
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

# Add significance stars
x_range = plot_df["standardized_difference"].max() - plot_df["standardized_difference"].min()
offset = 0.04 * x_range

for bar, value, stars in zip(
    bars,
    plot_df["standardized_difference"],
    plot_df["significance"]
):
    if stars != "":
        y = bar.get_y() + bar.get_height() / 2

        if value > 0:
            x = value + offset
            ha = "left"
        else:
            x = value - offset
            ha = "right"

        ax.text(
            x,
            y,
            stars,
            va="center",
            ha=ha,
            fontsize=10,
            fontweight="bold"
        )

ax.set_xlabel("Patient-level standardized mean difference")
ax.set_ylabel("Feature group")
ax.set_title("Patient-level AML-vs-control nuclear shape differences")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_elements = [
    Patch(facecolor="#B2182B", edgecolor="black", label="Higher in AML"),
    Patch(facecolor="#2166AC", edgecolor="black", label="Higher in control"),
]

ax.legend(
    handles=legend_elements,
    frameon=False,
    loc="lower right"
)

plt.tight_layout()

plt.savefig(
    output_dir / "patient_level_feature_group_standardized_difference_barplot_with_significance.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
#plot comparison showing each patient distribution
import matplotlib.pyplot as plt
import numpy as np

# patient_group_df should contain:
# patient, label_col, and GROUP_SCORE_COLS

plot_cols = GROUP_SCORE_COLS
# Fixed colors
control_color = "#2166AC"  # blue
aml_color = "#B2182B"      # red

long_df = patient_group_df.melt(
    id_vars=[patient_col, label_col],
    value_vars=plot_cols,
    var_name="feature_group",
    value_name="group_z_score"
)

pretty_names = {
    "group_z_size": "Size",
    "group_z_circularity": "Circularity",
    "group_z_solidity": "Solidity",
    "group_z_elongation_anisotropy": "Elongation / Anisotropy",
    "group_z_boundary_irregularity": "Boundary Irregularity",
    "group_z_texture_heterogeneity": "Texture Heterogeneity",
    "group_z_radial_intensity_distribution": "Radial Intensity Distribution",
    "group_z_intensity_asymmetry": "Intensity asymmetry"
}


long_df["feature_group_label"] = long_df["feature_group"].map(pretty_names)

fig, ax = plt.subplots(figsize=(4, 3.5))

positions = np.arange(len(order))

for i, group in enumerate(order):
    sub = long_df[long_df["feature_group_label"] == group]

    aml_vals = sub.loc[sub[label_col] == aml_label, "group_z_score"].dropna()
    ctrl_vals = sub.loc[sub[label_col] == control_label, "group_z_score"].dropna()

    # jitter patient points
    ax.scatter(
        np.random.normal(i - 0.12, 0.025, size=len(ctrl_vals)),
        ctrl_vals,
        s=35,
        alpha=0.8,
        color=control_color,
        label="Control" if i == 0 else None
    )

    ax.scatter(
        np.random.normal(i + 0.12, 0.025, size=len(aml_vals)),
        aml_vals,
        s=35,
        alpha=0.8,
        color=aml_color,
        label="AML" if i == 0 else None
    )

    # mean lines
    ax.hlines(
        ctrl_vals.mean(),
        i - 0.25,
        i,
        color="black",
        linewidth=2
    )

    ax.hlines(
        aml_vals.mean(),
        i,
        i + 0.25,
        color="black",
        linewidth=2
    )

ax.axhline(0, color="0.5", linewidth=1, linestyle="--")

ax.set_xticks(positions)
ax.set_xticklabels(order, rotation=35, ha="right")

ax.set_ylabel("Patient-level mean group z-score")
ax.set_xlabel("Feature group")

ax.set_title("Patient-level nuclear shape feature-group scores")

ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(output_dir /
    "patient_level_feature_group_scores_by_class.pdf",
    bbox_inches="tight"
)

plt.show()

###
At the patient level, none of the predefined nuclear-shape feature groups showed a statistically significant AML-versus-control difference after correction for multiple testing. Boundary roughness showed the largest standardized difference between groups, but this effect did not remain significant after FDR correction.
this is the Welch p-value with FDR correction, because we are comparing patient-level group means between two independent groups with unequal sample sizes (aml 33 and ct 15). The Mann–Whitney result can be mentioned as a robustness check, but avoid claiming significance for boundary roughness because its adjusted p-value is above 0.05.

# Correlation between features and probability 

In [ ]:
[c for c in df_merged.columns if "prob" in c.lower()]

In [ ]:
prob_col = "prob_positive"

In [ ]:
#test normality
patient_corr_df = (
    df_merged
    .groupby(patient_col)[[prob_col] + GROUP_SCORE_COLS]
    .mean()
    .reset_index()
)
from scipy import stats
import pandas as pd
import numpy as np

normality_results = []

cols_to_test = [prob_col] + GROUP_SCORE_COLS

for col in cols_to_test:
    values = patient_corr_df[col].dropna().astype(float)

    shapiro_stat, shapiro_p = stats.shapiro(values)

    normality_results.append({
        "variable": col,
        "n": len(values),
        "shapiro_W": shapiro_stat,
        "shapiro_p": shapiro_p,
        "normal_by_shapiro": shapiro_p >= 0.05
    })

normality_df = pd.DataFrame(normality_results)

normality_df.round(4)

In [ ]:
#Automatically choose Pearson or Spearman
#Pearson if both variables look approximately normal
#Spearman if either variable is non-normal

from scipy import stats
from statsmodels.stats.multitest import multipletests

correlation_results = []

# normality of AML probability
prob_values = patient_corr_df[prob_col].dropna().astype(float)
_, prob_normality_p = stats.shapiro(prob_values)
prob_is_normal = prob_normality_p >= 0.05

for col in GROUP_SCORE_COLS:
    tmp = patient_corr_df[[prob_col, col]].dropna().copy()

    x = tmp[prob_col].astype(float)
    y = tmp[col].astype(float)

    # normality of feature group
    _, feature_normality_p = stats.shapiro(y)
    feature_is_normal = feature_normality_p >= 0.05

    # choose correlation test
    if prob_is_normal and feature_is_normal:
        corr_method = "Pearson"
        r, p_value = stats.pearsonr(x, y)
    else:
        corr_method = "Spearman"
        r, p_value = stats.spearmanr(x, y)

    correlation_results.append({
        "feature_group": col.replace("group_z_", ""),
        "correlation_method": corr_method,
        "correlation": r,
        "p_value": p_value,
        "AML_probability_shapiro_p": prob_normality_p,
        "feature_group_shapiro_p": feature_normality_p,
        "AML_probability_normal": prob_is_normal,
        "feature_group_normal": feature_is_normal,
        "n_patients": len(tmp)
    })

patient_feature_group_corr_df = pd.DataFrame(correlation_results)

# FDR correction across feature groups
patient_feature_group_corr_df["p_adj_BH"] = multipletests(
    patient_feature_group_corr_df["p_value"],
    method="fdr_bh"
)[1]

patient_feature_group_corr_df = patient_feature_group_corr_df.sort_values(
    "correlation"
)

patient_feature_group_corr_df.to_csv(output_dir /
    "AML_vs_CTRL_AINUposprob_feature_correlation_startistics.csv",
    index=False
)

patient_feature_group_corr_df.round(4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

plot_df = patient_feature_group_corr_df.copy()

#label_map = {
#    "size": "Size",
#    "roundness_circularity": "Roundness / circularity",
#    "elongation": "Elongation",
#    "boundary_roughness": "Boundary roughness",
#    "mass_distribution": "Mass distribution",
#}

plot_df["feature_group_label"] = (
    plot_df["feature_group"]
    .map(label_map)
    .fillna(plot_df["feature_group"])
)

plot_df = plot_df.sort_values("correlation")

colors = np.where(
    plot_df["correlation"] > 0,
    "#B2182B",
    "#2166AC"
)

fig, ax = plt.subplots(figsize=(5.8, 3.4))

ax.barh(
    plot_df["feature_group_label"],
    plot_df["correlation"],
    color=colors,
    edgecolor="black",
    linewidth=0.6
)

ax.axvline(0, color="black", linewidth=1)

ax.set_xlabel("Correlation with AML probability")
ax.set_ylabel("Feature group")
ax.set_title("Patient-level association with AML probability")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_elements = [
    Patch(facecolor="#B2182B", edgecolor="black", label="Positive correlation"),
    Patch(facecolor="#2166AC", edgecolor="black", label="Negative correlation"),
]

ax.legend(handles=legend_elements, frameon=False, loc="lower right")

# Add FDR significance stars
for i, (_, row) in enumerate(plot_df.iterrows()):
    x = row["correlation"]
    p = row["p_adj_BH"]

    if p < 0.001:
        star = "***"
    elif p < 0.01:
        star = "**"
    elif p < 0.05:
        star = "*"
    else:
        star = ""

    if star:
        offset = 0.015 if x >= 0 else -0.015
        ha = "left" if x >= 0 else "right"

        ax.text(
            x + offset,
            i,
            star,
            va="center",
            ha=ha,
            fontsize=11
        )

plt.tight_layout()

plt.savefig(output_dir /
    "patient_level_feature_group_correlation_with_AML_probability.pdf",
    bbox_inches="tight"
)

plt.show()

### summary 
To evaluate whether predefined nuclear-shape feature groups were associated with model-derived AML probability, individual descriptors were standardized and averaged within each feature group, and both feature-group scores and AML probability were summarized at the patient level. Normality was assessed using the Shapiro-Wilk test, which indicated that AML probability was not normally distributed across patients; therefore, Spearman rank correlations were used. Among the five feature groups, boundary roughness showed the strongest association with AML probability, with a negative correlation (Spearman r = -0.343, nominal p = 0.017). However, this association did not remain significant after Benjamini-Hochberg correction for multiple testing (FDR-adjusted p = 0.086). No other feature group showed a significant association with AML probability.

Interpretation: there is some evidence of a trend linking boundary roughness to AML probability, but statistically you should report it as not significant after multiple-testing correction.

# Filter by true labelled cells

In [ ]:
# Predicted class from model probability
df_merged["predicted_label"] = np.where(
    df_merged[prob_col] >= 0.5,
    "AML",
    "Control"
)

# Whether each single file/cell was correctly classified
df_merged["correct"] = df_merged["predicted_label"] == df_merged[label_col]

# New dataframe containing only correctly classified files/cells
correct_df = df_merged[df_merged["correct"] == True].copy()

print(correct_df.shape)
correct_df

In [ ]:
# Use only correctly classified files/cells
plot_df = correct_df.copy()

# Optional: make sure labels are clean
plot_df[label_col] = plot_df[label_col].astype(str)

# Number of feature groups
n_groups = len(FEATURE_GROUPS)

# Arrange plots
ncols = 4
nrows = int(np.ceil(n_groups / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(4.2 * ncols, 3.6 * nrows),
    sharey=True
)

axes = axes.flatten()

for ax, (group_name, features) in zip(axes, FEATURE_GROUPS.items()):

    # Keep only features that are actually present in the dataframe
    available_features = [f for f in features if f in plot_df.columns]

    if len(available_features) == 0:
        ax.set_title(f"{group_name}\n(no features found)")
        ax.axis("off")
        continue

    # Create one value per feature group by averaging features in that group
    plot_df[group_name + "_score"] = plot_df[available_features].mean(axis=1)

    sns.scatterplot(
        data=plot_df,
        x=group_name + "_score",
        y=prob_col,
        hue=label_col,
        alpha=0.7,
        s=35,
        ax=ax
    )

    ax.set_title(group_name)
    ax.set_xlabel("Mean feature-group value")
    ax.set_ylabel("Positive probability")

    ax.axhline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.6)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Remove repeated legends from each subplot
    ax.get_legend().remove()

# Turn off empty axes if any
for ax in axes[n_groups:]:
    ax.axis("off")

# Add one shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title=label_col,
    loc="upper center",
    ncol=len(labels),
    bbox_to_anchor=(0.5, 1.02)
)

fig.suptitle(
    "Correctly classified files: positive probability vs feature-group score",
    y=1.06,
    fontsize=14
)

plt.tight_layout()

plt.savefig(output_dir /
    "scatterplot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Flatten FEATURE_GROUPS into one feature list
selected_features = [
    feature
    for group, features in FEATURE_GROUPS.items()
    for feature in features
]

print("Features used for PCA:")
for f in selected_features:
    print(f)

# Use selected features + label column
plot_cols = selected_features + [label_col]

pca_df = df_merged.loc[:, plot_cols].copy()

# Remove rows with missing feature values or missing labels
pca_df = pca_df.dropna(subset=selected_features + [label_col])

X = pca_df[selected_features].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
pca_values = pca.fit_transform(X_scaled)

pca_df["PC1"] = pca_values[:, 0]
pca_df["PC2"] = pca_values[:, 1]

# Plot PCA colored by new_labels_A
plt.figure(figsize=(7, 6))

for label in pca_df[label_col].unique():
    subset = pca_df[pca_df[label_col] == label]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
plt.title("PCA of selected nuclear features")
plt.legend(title=label_col)
plt.savefig(output_dir /
    "PCA_features.pdf",
    bbox_inches="tight"
)
plt.tight_layout()
plt.show()


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=selected_features,
    columns=["PC1_loading", "PC2_loading"]
)

print("Top PC1 contributors:")
print(loadings.reindex(loadings["PC1_loading"].abs().sort_values(ascending=False).index))

print("\nTop PC2 contributors:")
print(loadings.reindex(loadings["PC2_loading"].abs().sort_values(ascending=False).index))

In [ ]:
# ============================================================
# t-SNE plot using selected grouped nuclear features
# Colored by label_col
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Flatten FEATURE_GROUPS into one feature list
selected_features = [
    feature
    for group, features in FEATURE_GROUPS.items()
    for feature in features
]

print("Features used for t-SNE:")
for f in selected_features:
    print(f)

# Use selected features + label column
plot_cols = selected_features + [label_col]

tsne_df = df_merged.loc[:, plot_cols].copy()

# Remove rows with missing feature values or missing labels
tsne_df = tsne_df.dropna(subset=selected_features + [label_col])

X = tsne_df[selected_features].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

tsne_values = tsne.fit_transform(X_scaled)

tsne_df["tSNE1"] = tsne_values[:, 0]
tsne_df["tSNE2"] = tsne_values[:, 1]

# Plot t-SNE colored by new_labels_A
plt.figure(figsize=(7, 6))

for label in tsne_df[label_col].unique():
    subset = tsne_df[tsne_df[label_col] == label]

    plt.scatter(
        subset["tSNE1"],
        subset["tSNE2"],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of selected nuclear features")
plt.legend(title=label_col)
plt.savefig(output_dir /
    "t-SNE_features.pdf",
    bbox_inches="tight"
)
plt.tight_layout()
plt.show()